In [2]:
import json
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_53"
API_KEY = "oc_wP6hi7abKU3n6IeFV8sZLjiOA3TSMmyX"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode(),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY
        }
    )
    with urlopen(req, timeout=30) as r:
        return json.load(r)

spec = api({
    "action": "spec",
    "team_id": TEAM_ID
})

SPACE = spec["hyperparameters"]

print("Model:", spec["model"])

for name, values in SPACE.items():
    print(name, ":", values)

Model: ElasticNet
tol : [0.001, 0.0001]
alpha : [0.0001, 0.00020691380811147902, 0.00042813323987193956, 0.0008858667904100822, 0.0018329807108324356, 0.00379269019073225, 0.007847599703514606, 0.01623776739188721, 0.03359818286283781, 0.06951927961775606, 0.14384498882876628, 0.29763514416313164, 0.615848211066026, 1.2742749857031321, 2.6366508987303554, 5.455594781168514, 11.288378916846883, 23.357214690901213, 48.32930238571752, 100]
l1_ratio : [0.05, 0.17857142857142855, 0.3071428571428571, 0.43571428571428567, 0.5642857142857143, 0.6928571428571428, 0.8214285714285714, 0.95]
positive : [True, False]
selection : ['cyclic', 'random']
fit_intercept : [True, False]


In [3]:
TEAM_ID = "TEAM_53"
API_KEY = "oc_wP6hi7abKU3n6IeFV8sZLjiOA3TSMmyX"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

In [5]:
def oracle_query(params):
    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })
    return float(result["loss"])

In [6]:
results = []

In [7]:
def query_and_store(params):
    for r in results:
        if r["params"] == params:
            print("Already queried. Skipping.")
            return r["loss"]

    loss = oracle_query(params)

    results.append({
        "params": params.copy(),
        "loss": loss
    })

    print("Oracle call:", len(results))
    print("Loss:", loss)

    return loss

In [8]:
params = {
    "tol": 0.0001,
    "alpha": 0.007847599703514606,
    "l1_ratio": 0.5642857142857143,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 1
Loss: 10.988421278696


10.988421278696

In [9]:
params = {
    "tol": 0.0001,
    "alpha": 0.0008858667904100822,
    "l1_ratio": 0.5642857142857143,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 2
Loss: 11.0036458740899


11.0036458740899

In [10]:
alpha_test = [
    0.0001,
    0.0008858667904100822,
    0.007847599703514606,
    0.06951927961775606,
    0.615848211066026,
    5.455594781168514,
    100
]

l1_test = [
    0.05,
    0.3071428571428571,
    0.5642857142857143,
    0.8214285714285714,
    0.95
]

for alpha in alpha_test:
    for l1_ratio in l1_test:

        params = {
            "tol": 0.0001,
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "positive": False,
            "selection": "cyclic",
            "fit_intercept": True
        }

        query_and_store(params)

print("Total Oracle calls:", len(results))

Oracle call: 3
Loss: 11.0055947597813
Oracle call: 4
Loss: 11.0055811436388
Oracle call: 5
Loss: 11.0055685071715
Oracle call: 6
Loss: 11.0055558981696
Oracle call: 7
Loss: 11.0055496039697
Oracle call: 8
Loss: 11.0038822314091
Oracle call: 9
Loss: 11.0037574805709
Already queried. Skipping.
Oracle call: 10
Loss: 11.0035364231637
Oracle call: 11
Loss: 11.0034825070515
Oracle call: 12
Loss: 10.9898716447207
Oracle call: 13
Loss: 10.9887448992272
Already queried. Skipping.
Oracle call: 14
Loss: 10.9886597319872
Oracle call: 15
Loss: 10.9888416237953
Oracle call: 16
Loss: 10.9486672827174
Oracle call: 17
Loss: 10.9477484316569
Oracle call: 18
Loss: 10.9572802414266
Oracle call: 19
Loss: 10.9802728504869
Oracle call: 20
Loss: 10.986635338665
Oracle call: 21
Loss: 12.7522140158236
Oracle call: 22
Loss: 12.5568403956744
Oracle call: 23
Loss: 12.4281039968251
Oracle call: 24
Loss: 12.2759666139604
Oracle call: 25
Loss: 12.2088695201524
Oracle call: 26
Loss: 21.2118739452028
Oracle call: 27
Lo

In [11]:
best_results = sorted(results, key=lambda x: x["loss"])

for i, r in enumerate(best_results[:10], start=1):
    print(f"Rank {i}")
    print("Loss:", r["loss"])
    print("Params:", r["params"])
    print()

Rank 1
Loss: 10.9477484316569
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 2
Loss: 10.9486672827174
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 3
Loss: 10.9572802414266
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 4
Loss: 10.9802728504869
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.8214285714285714, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 5
Loss: 10.986635338665
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.95, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 6
Loss: 10.988421278696
Params: {'tol': 0.0001, 'alpha': 0.007847599703514606, 'l1_ratio': 0.5642857142857143, 'po

In [12]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 36
Loss: 10.9476606732259


10.9476606732259

In [13]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.3071428571428571,
    "positive": True,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 37
Loss: 10.966799765597


10.966799765597

In [14]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "random",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 38
Loss: 10.9493531440032


10.9493531440032

In [15]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": False
}

query_and_store(params)

Oracle call: 39
Loss: 226.585383366762


226.585383366762

In [16]:
params = {
    "tol": 0.001,
    "alpha": 0.14384498882876628,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 40
Loss: 11.0467864570773


11.0467864570773

In [17]:
params = {
    "tol": 0.001,
    "alpha": 0.007847599703514606,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 41
Loss: 10.9886346157953


10.9886346157953

In [18]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.17857142857142855,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 42
Loss: 10.9494865278444


10.9494865278444

In [19]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.43571428571428567,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 43
Loss: 10.94928593065


10.94928593065

In [20]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "random",
    "fit_intercept": True
}

query_and_store(params)

Already queried. Skipping.


10.9493531440032

In [21]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.5642857142857143,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 44
Loss: 10.9572196852644


10.9572196852644

In [22]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.6928571428571428,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 45
Loss: 10.9708465282779


10.9708465282779

In [23]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 46
Loss: 10.9486038646824


10.9486038646824

In [24]:
params = {
    "tol": 0.001,
    "alpha": 0.01623776739188721,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 47
Loss: 10.9753179144396


10.9753179144396

In [25]:
params = {
    "tol": 0.001,
    "alpha": 0.29763514416313164,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 48
Loss: 11.4170035864397


11.4170035864397

In [26]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.8214285714285714,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 49
Loss: 10.9801777640171


10.9801777640171

In [27]:
tested = {
    tuple(sorted(r["params"].items()))
    for r in results
}

candidates = []

for tol in SPACE["tol"]:
    for alpha in SPACE["alpha"]:
        for l1_ratio in SPACE["l1_ratio"]:
            for positive in SPACE["positive"]:
                for selection in SPACE["selection"]:
                    for fit_intercept in SPACE["fit_intercept"]:

                        params = {
                            "tol": tol,
                            "alpha": alpha,
                            "l1_ratio": l1_ratio,
                            "positive": positive,
                            "selection": selection,
                            "fit_intercept": fit_intercept
                        }

                        if tuple(sorted(params.items())) not in tested:
                            candidates.append(params)

print("Total possible configurations:", 1280)
print("Already tested:", len(results))
print("Remaining:", len(candidates))

Total possible configurations: 1280
Already tested: 49
Remaining: 2511


In [28]:
for name, values in SPACE.items():
    print(name, len(values))

total = 1
for values in SPACE.values():
    total *= len(values)

print("Actual total configurations:", total)
print("Recorded Oracle calls:", len(results))

tol 2
alpha 20
l1_ratio 8
positive 2
selection 2
fit_intercept 2
Actual total configurations: 2560
Recorded Oracle calls: 49


In [29]:
best_results = sorted(results, key=lambda x: x["loss"])

for i, r in enumerate(best_results[:10], start=1):
    print(f"Rank {i}")
    print("Loss:", r["loss"])
    print("Params:", r["params"])
    print()

Rank 1
Loss: 10.9476606732259
Params: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 2
Loss: 10.9477484316569
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 3
Loss: 10.9486038646824
Params: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 4
Loss: 10.9486672827174
Params: {'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 5
Loss: 10.94928593065
Params: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.43571428571428567, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 6
Loss: 10.9493531440032
Params: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positi

In [34]:
print("Recorded Oracle calls:", len(results))

best = min(results, key=lambda x: x["loss"])

print("Best loss:", best["loss"])
print("Best params:", best["params"])

Recorded Oracle calls: 50
Best loss: 10.9476606732259
Best params: {'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}


In [33]:
params = {
    "tol": 0.001,
    "alpha": 0.06951927961775606,
    "l1_ratio": 0.95,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 50
Loss: 10.9865910934823


10.9865910934823

In [35]:
params = {
    "tol": 0.001,
    "alpha": 0.03359818286283781,
    "l1_ratio": 0.3071428571428571,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 51
Loss: 10.9564177099459


10.9564177099459

In [36]:
params = {
    "tol": 0.001,
    "alpha": 0.03359818286283781,
    "l1_ratio": 0.05,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 52
Loss: 10.9569195602737


10.9569195602737

In [37]:
params = {
    "tol": 0.001,
    "alpha": 0.03359818286283781,
    "l1_ratio": 0.43571428571428567,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

query_and_store(params)

Oracle call: 53
Loss: 10.9564732024813


10.9564732024813

In [38]:
tested = {
    tuple(sorted(r["params"].items()))
    for r in results
}

remaining = []

for tol in SPACE["tol"]:
    for alpha in SPACE["alpha"]:
        for l1_ratio in SPACE["l1_ratio"]:
            for positive in SPACE["positive"]:
                for selection in SPACE["selection"]:
                    for fit_intercept in SPACE["fit_intercept"]:
                        params = {
                            "tol": tol,
                            "alpha": alpha,
                            "l1_ratio": l1_ratio,
                            "positive": positive,
                            "selection": selection,
                            "fit_intercept": fit_intercept
                        }

                        if tuple(sorted(params.items())) not in tested:
                            remaining.append(params)

print("Total configurations:", 2560)
print("Oracle calls used:", len(results))
print("Remaining:", len(remaining))

Total configurations: 2560
Oracle calls used: 53
Remaining: 2507


In [39]:
from collections import defaultdict

alpha_results = defaultdict(list)

for r in results:
    alpha = r["params"]["alpha"]
    alpha_results[alpha].append(r["loss"])

alpha_summary = []

for alpha, losses in alpha_results.items():
    alpha_summary.append({
        "alpha": alpha,
        "best_loss": min(losses),
        "tests": len(losses)
    })

alpha_summary = sorted(alpha_summary, key=lambda x: x["best_loss"])

for row in alpha_summary:
    print(
        "alpha =", row["alpha"],
        "| best loss =", row["best_loss"],
        "| tests =", row["tests"]
    )

alpha = 0.06951927961775606 | best loss = 10.9476606732259 | tests = 16
alpha = 0.03359818286283781 | best loss = 10.9564177099459 | tests = 3
alpha = 0.01623776739188721 | best loss = 10.9753179144396 | tests = 1
alpha = 0.007847599703514606 | best loss = 10.988421278696 | tests = 6
alpha = 0.0008858667904100822 | best loss = 11.0034825070515 | tests = 5
alpha = 0.0001 | best loss = 11.0055496039697 | tests = 5
alpha = 0.14384498882876628 | best loss = 11.0467864570773 | tests = 1
alpha = 0.29763514416313164 | best loss = 11.4170035864397 | tests = 1
alpha = 0.615848211066026 | best loss = 12.2088695201524 | tests = 5
alpha = 5.455594781168514 | best loss = 21.2118739452028 | tests = 5
alpha = 100 | best loss = 25.0919434815282 | tests = 5


In [40]:
from collections import defaultdict

binary_results = defaultdict(list)

for r in results:
    p = r["params"]

    key = (
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    )

    binary_results[key].append(r["loss"])

binary_summary = []

for key, losses in binary_results.items():
    binary_summary.append({
        "tol": key[0],
        "positive": key[1],
        "selection": key[2],
        "fit_intercept": key[3],
        "best_loss": min(losses),
        "tests": len(losses)
    })

binary_summary = sorted(
    binary_summary,
    key=lambda x: x["best_loss"]
)

for row in binary_summary:
    print(row)

{'tol': 0.001, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True, 'best_loss': 10.9476606732259, 'tests': 15}
{'tol': 0.0001, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True, 'best_loss': 10.9477484316569, 'tests': 35}
{'tol': 0.001, 'positive': False, 'selection': 'random', 'fit_intercept': True, 'best_loss': 10.9493531440032, 'tests': 1}
{'tol': 0.001, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True, 'best_loss': 10.966799765597, 'tests': 1}
{'tol': 0.001, 'positive': False, 'selection': 'cyclic', 'fit_intercept': False, 'best_loss': 226.585383366762, 'tests': 1}


In [41]:
tested_alphas = set()

for r in results:
    p = r["params"]

    if (
        p["tol"] == 0.001
        and p["positive"] == False
        and p["selection"] == "cyclic"
        and p["fit_intercept"] == True
    ):
        tested_alphas.add(p["alpha"])

best_alpha = 0.06951927961775606

print("Tested alpha values:")
for a in sorted(tested_alphas):
    print(a)

print("\nUntested alpha values:")
for a in sorted(SPACE["alpha"]):
    if a not in tested_alphas:
        print(a, "| distance from best alpha:", abs(a - best_alpha))

Tested alpha values:
0.007847599703514606
0.01623776739188721
0.03359818286283781
0.06951927961775606
0.14384498882876628
0.29763514416313164

Untested alpha values:
0.0001 | distance from best alpha: 0.06941927961775606
0.00020691380811147902 | distance from best alpha: 0.06931236580964457
0.00042813323987193956 | distance from best alpha: 0.06909114637788412
0.0008858667904100822 | distance from best alpha: 0.06863341282734597
0.0018329807108324356 | distance from best alpha: 0.06768629890692363
0.00379269019073225 | distance from best alpha: 0.06572658942702381
0.615848211066026 | distance from best alpha: 0.54632893144827
1.2742749857031321 | distance from best alpha: 1.204755706085376
2.6366508987303554 | distance from best alpha: 2.5671316191125992
5.455594781168514 | distance from best alpha: 5.386075501550758
11.288378916846883 | distance from best alpha: 11.218859637229126
23.357214690901213 | distance from best alpha: 23.287695411283458
48.32930238571752 | distance from best 

In [42]:
pair_results = {}

for r in results:
    p = r["params"]

    key = (p["alpha"], p["l1_ratio"])

    if key not in pair_results or r["loss"] < pair_results[key]:
        pair_results[key] = r["loss"]

pair_summary = sorted(
    pair_results.items(),
    key=lambda x: x[1]
)

print("Top alpha + l1_ratio pairs:")

for i, ((alpha, l1_ratio), loss) in enumerate(pair_summary[:15], 1):
    print(
        i,
        "| alpha =", alpha,
        "| l1_ratio =", l1_ratio,
        "| best loss =", loss
    )

Top alpha + l1_ratio pairs:
1 | alpha = 0.06951927961775606 | l1_ratio = 0.3071428571428571 | best loss = 10.9476606732259
2 | alpha = 0.06951927961775606 | l1_ratio = 0.05 | best loss = 10.9486038646824
3 | alpha = 0.06951927961775606 | l1_ratio = 0.43571428571428567 | best loss = 10.94928593065
4 | alpha = 0.06951927961775606 | l1_ratio = 0.17857142857142855 | best loss = 10.9494865278444
5 | alpha = 0.03359818286283781 | l1_ratio = 0.3071428571428571 | best loss = 10.9564177099459
6 | alpha = 0.03359818286283781 | l1_ratio = 0.43571428571428567 | best loss = 10.9564732024813
7 | alpha = 0.03359818286283781 | l1_ratio = 0.05 | best loss = 10.9569195602737
8 | alpha = 0.06951927961775606 | l1_ratio = 0.5642857142857143 | best loss = 10.9572196852644
9 | alpha = 0.06951927961775606 | l1_ratio = 0.6928571428571428 | best loss = 10.9708465282779
10 | alpha = 0.01623776739188721 | l1_ratio = 0.3071428571428571 | best loss = 10.9753179144396
11 | alpha = 0.06951927961775606 | l1_ratio = 0.

In [43]:
best = min(results, key=lambda x: x["loss"])

print("FINAL BEST OBSERVED")
print("-------------------")
print("Loss:", best["loss"])

for key, value in best["params"].items():
    print(f"{key}: {value}")

print("\nTotal Oracle calls:", len(results))
print("Total configurations:", 2560)
print("Configurations remaining:", 2560 - len(results))

FINAL BEST OBSERVED
-------------------
Loss: 10.9476606732259
tol: 0.001
alpha: 0.06951927961775606
l1_ratio: 0.3071428571428571
positive: False
selection: cyclic
fit_intercept: True

Total Oracle calls: 53
Total configurations: 2560
Configurations remaining: 2507


In [44]:
from collections import defaultdict

binary_groups = defaultdict(list)

for r in results:
    p = r["params"]

    key = (
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    )

    binary_groups[key].append(r)

print("Binary configuration coverage:")
print()

for key, vals in sorted(binary_groups.items()):
    print(
        f"tol={key[0]} | "
        f"positive={key[1]} | "
        f"selection={key[2]} | "
        f"fit_intercept={key[3]}"
    )
    print("  Tested:", len(vals))
    print("  Remaining:", 160 - len({
        (r["params"]["alpha"], r["params"]["l1_ratio"])
        for r in vals
    }))

Binary configuration coverage:

tol=0.0001 | positive=False | selection=cyclic | fit_intercept=True
  Tested: 35
  Remaining: 125
tol=0.001 | positive=False | selection=cyclic | fit_intercept=False
  Tested: 1
  Remaining: 159
tol=0.001 | positive=False | selection=cyclic | fit_intercept=True
  Tested: 15
  Remaining: 145
tol=0.001 | positive=False | selection=random | fit_intercept=True
  Tested: 1
  Remaining: 159
tol=0.001 | positive=True | selection=cyclic | fit_intercept=True
  Tested: 1
  Remaining: 159


In [45]:
import itertools

binary_space = list(itertools.product(
    SPACE["tol"],
    SPACE["positive"],
    SPACE["selection"],
    SPACE["fit_intercept"]
))

tested_binary = set()

for r in results:
    p = r["params"]
    tested_binary.add((
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    ))

unexplored_binary = [
    b for b in binary_space
    if b not in tested_binary
]

alpha_indices = [0, 3, 6, 8, 9, 10, 14, 19]
explore_alphas = [SPACE["alpha"][i] for i in alpha_indices]

exploration_plan = []

for tol, positive, selection, fit_intercept in unexplored_binary:
    for alpha in explore_alphas:
        for l1_ratio in SPACE["l1_ratio"]:
            exploration_plan.append({
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1_ratio,
                "positive": positive,
                "selection": selection,
                "fit_intercept": fit_intercept
            })

print("Unexplored combinations:", len(unexplored_binary))
print("Planned calls:", len(exploration_plan))
print("Total after:", len(results) + len(exploration_plan))

Unexplored combinations: 11
Planned calls: 704
Total after: 757


In [59]:
batch = exploration_plan[:100]

for params in batch:
    query_and_store(params)

print("Total calls:", len(results))

Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
Already queried. Skipping.
A

In [47]:
print("Oracle calls:", len(results))

Oracle calls: 95


In [48]:
remaining_plan = [
    p for p in exploration_plan
    if not any(r["params"] == p for r in results)
]

print("Remaining planned calls:", len(remaining_plan))
print("Current Oracle calls:", len(results))

Remaining planned calls: 662
Current Oracle calls: 95


In [49]:
batch = remaining_plan[:25]

for params in batch:
    try:
        query_and_store(params)
    except Exception as e:
        print("Request failed:", type(e).__name__)
        break

print("Oracle calls:", len(results))

Oracle call: 96
Loss: 227.447213974393
Oracle call: 97
Loss: 227.24684108811
Request failed: TimeoutError
Oracle calls: 97


In [58]:
for params in remaining_plan[:25]:
    try:
        query_and_store(params)
    except Exception as e:
        print("Request failed:", type(e).__name__)
        break

remaining_plan = [
    p for p in exploration_plan
    if not any(r["params"] == p for r in results)
]

print("Oracle calls:", len(results))

Oracle call: 146
Loss: 11.0054620029697
Oracle call: 147
Loss: 11.0054585293872
Oracle call: 148
Loss: 11.0054550621217
Oracle call: 149
Loss: 11.0054516011737
Oracle call: 150
Loss: 11.0054481465438
Oracle call: 151
Loss: 11.0043314795057
Oracle call: 152
Loss: 11.0042990938475
Oracle call: 153
Loss: 11.004267202481
Oracle call: 154
Loss: 11.0042696999988
Oracle call: 155
Loss: 11.0042388207395
Oracle call: 156
Loss: 11.0042084368735
Oracle call: 157
Loss: 11.004178548777
Oracle call: 158
Loss: 11.0041491568265
Oracle call: 159
Loss: 10.9952748319026
Oracle call: 160
Loss: 10.9948784478631
Oracle call: 161
Loss: 10.9946778352482
Oracle call: 162
Loss: 10.994357129654
Oracle call: 163
Loss: 10.9940748278477
Oracle call: 164
Loss: 10.9938311879071
Oracle call: 165
Loss: 10.9936264696772
Oracle call: 166
Loss: 10.9934609347832
Oracle call: 167
Loss: 10.9766225509629
Oracle call: 168
Loss: 10.9734231615866
Oracle call: 169
Loss: 10.9708641416964
Oracle call: 170
Loss: 10.9689635107327
Ora

In [51]:
from collections import defaultdict

grouped = defaultdict(list)

for r in results:
    p = r["params"]
    key = (
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    )
    grouped[key].append(r["loss"])

summary = []

for key, losses in grouped.items():
    summary.append((
        min(losses),
        len(losses),
        key
    ))

summary.sort()

for best_loss, count, key in summary:
    print(
        key,
        "| best loss =", best_loss,
        "| tests =", count
    )

(0.001, False, 'cyclic', True) | best loss = 10.9476606732259 | tests = 15
(0.0001, False, 'cyclic', True) | best loss = 10.9477484316569 | tests = 35
(0.001, False, 'random', True) | best loss = 10.9493531440032 | tests = 1
(0.001, True, 'cyclic', True) | best loss = 10.966799765597 | tests = 1
(0.001, True, 'random', True) | best loss = 11.0054488256388 | tests = 3
(0.001, True, 'cyclic', False) | best loss = 225.404278818708 | tests = 64
(0.001, False, 'cyclic', False) | best loss = 226.585383366762 | tests = 1


In [52]:
binary_space = list(itertools.product(
    SPACE["tol"],
    SPACE["positive"],
    SPACE["selection"],
    SPACE["fit_intercept"]
))

tested_binary = set()

for r in results:
    p = r["params"]
    tested_binary.add((
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    ))

unexplored_binary = [
    b for b in binary_space
    if b not in tested_binary
]

print("Unexplored binary combinations:", len(unexplored_binary))

for b in unexplored_binary:
    print(b)

Unexplored binary combinations: 9
(0.001, True, 'random', False)
(0.001, False, 'random', False)
(0.0001, True, 'cyclic', True)
(0.0001, True, 'cyclic', False)
(0.0001, True, 'random', True)
(0.0001, True, 'random', False)
(0.0001, False, 'cyclic', False)
(0.0001, False, 'random', True)
(0.0001, False, 'random', False)


In [53]:
explore_l1 = [
    0.05,
    0.3071428571428571
]

global_exploration = []

for tol, positive, selection, fit_intercept in unexplored_binary:
    for alpha in SPACE["alpha"]:
        for l1_ratio in explore_l1:
            global_exploration.append({
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1_ratio,
                "positive": positive,
                "selection": selection,
                "fit_intercept": fit_intercept
            })

global_exploration = [
    p for p in global_exploration
    if not any(r["params"] == p for r in results)
]

print("Planned calls:", len(global_exploration))

Planned calls: 360


In [54]:
for params in global_exploration[:25]:
    try:
        query_and_store(params)
    except Exception as e:
        print("Request failed:", type(e).__name__)
        break

global_exploration = [
    p for p in global_exploration
    if not any(r["params"] == p for r in results)
]

print("Oracle calls:", len(results))
print("Remaining exploration:", len(global_exploration))

Oracle call: 121
Loss: 225.404600495855
Oracle call: 122
Loss: 225.404505017724
Oracle call: 123
Loss: 225.406524406367
Oracle call: 124
Loss: 225.40671260986
Oracle call: 125
Loss: 225.410504945749
Oracle call: 126
Loss: 225.410481944109
Oracle call: 127
Loss: 225.419124777246
Oracle call: 128
Loss: 225.418280951819
Oracle call: 129
Loss: 225.436157364709
Oracle call: 130
Loss: 225.434303109763
Oracle call: 131
Loss: 225.471376393586
Oracle call: 132
Loss: 225.467595926567
Oracle call: 133
Loss: 225.544145509428
Oracle call: 134
Loss: 225.536649588543
Oracle call: 135
Loss: 225.694057826292
Oracle call: 136
Loss: 225.679200336956
Oracle call: 137
Loss: 226.002519849893
Oracle call: 138
Loss: 225.973989499169
Oracle call: 139
Loss: 226.64141609947
Oracle call: 140
Loss: 226.575550536538
Oracle call: 141
Loss: 227.884092593713
Oracle call: 142
Loss: 227.448636229429
Oracle call: 143
Loss: 229.957876960332
Oracle call: 144
Loss: 229.214898796847
Oracle call: 145
Loss: 233.41596444299
Ora

In [55]:
from collections import defaultdict

grouped = defaultdict(list)

for r in results:
    p = r["params"]
    key = (
        p["tol"],
        p["positive"],
        p["selection"],
        p["fit_intercept"]
    )
    grouped[key].append(r["loss"])

for key, losses in sorted(grouped.items(), key=lambda x: min(x[1])):
    print(
        key,
        "| best =", min(losses),
        "| calls =", len(losses)
    )

(0.001, False, 'cyclic', True) | best = 10.9476606732259 | calls = 15
(0.0001, False, 'cyclic', True) | best = 10.9477484316569 | calls = 35
(0.001, False, 'random', True) | best = 10.9493531440032 | calls = 1
(0.001, True, 'cyclic', True) | best = 10.966799765597 | calls = 1
(0.001, True, 'random', True) | best = 11.0054488256388 | calls = 3
(0.001, True, 'cyclic', False) | best = 225.404278818708 | calls = 64
(0.001, True, 'random', False) | best = 225.404505017724 | calls = 25
(0.001, False, 'cyclic', False) | best = 226.585383366762 | calls = 1


In [56]:
best_binary = {
    "tol": 0.001,
    "positive": False,
    "selection": "cyclic",
    "fit_intercept": True
}

tested = set()

for r in results:
    p = r["params"]
    if (
        p["tol"] == best_binary["tol"] and
        p["positive"] == best_binary["positive"] and
        p["selection"] == best_binary["selection"] and
        p["fit_intercept"] == best_binary["fit_intercept"]
    ):
        tested.add((p["alpha"], p["l1_ratio"]))

remaining = []

for alpha in SPACE["alpha"]:
    for l1_ratio in SPACE["l1_ratio"]:
        if (alpha, l1_ratio) not in tested:
            remaining.append((alpha, l1_ratio))

print("Tested:", len(tested))
print("Remaining:", len(remaining))

for alpha, l1_ratio in remaining:
    print(alpha, "|", l1_ratio)

Tested: 15
Remaining: 145
0.0001 | 0.05
0.0001 | 0.17857142857142855
0.0001 | 0.3071428571428571
0.0001 | 0.43571428571428567
0.0001 | 0.5642857142857143
0.0001 | 0.6928571428571428
0.0001 | 0.8214285714285714
0.0001 | 0.95
0.00020691380811147902 | 0.05
0.00020691380811147902 | 0.17857142857142855
0.00020691380811147902 | 0.3071428571428571
0.00020691380811147902 | 0.43571428571428567
0.00020691380811147902 | 0.5642857142857143
0.00020691380811147902 | 0.6928571428571428
0.00020691380811147902 | 0.8214285714285714
0.00020691380811147902 | 0.95
0.00042813323987193956 | 0.05
0.00042813323987193956 | 0.17857142857142855
0.00042813323987193956 | 0.3071428571428571
0.00042813323987193956 | 0.43571428571428567
0.00042813323987193956 | 0.5642857142857143
0.00042813323987193956 | 0.6928571428571428
0.00042813323987193956 | 0.8214285714285714
0.00042813323987193956 | 0.95
0.0008858667904100822 | 0.05
0.0008858667904100822 | 0.17857142857142855
0.0008858667904100822 | 0.3071428571428571
0.000885

In [61]:
promising_binary = [
    (0.001, False, "cyclic", True),
    (0.0001, False, "cyclic", True),
    (0.001, False, "random", True),
    (0.001, True, "cyclic", True),
    (0.001, True, "random", True)
]

smart_plan = []

for tol, positive, selection, fit_intercept in promising_binary:
    for alpha in SPACE["alpha"]:
        for l1_ratio in SPACE["l1_ratio"]:
            params = {
                "tol": tol,
                "alpha": alpha,
                "l1_ratio": l1_ratio,
                "positive": positive,
                "selection": selection,
                "fit_intercept": fit_intercept
            }

            if not any(r["params"] == params for r in results):
                smart_plan.append(params)

print("Smart candidates remaining:", len(smart_plan))
print("Current calls:", len(results))

Smart candidates remaining: 687
Current calls: 203


In [60]:
for params in smart_plan[:25]:
    try:
        query_and_store(params)
    except Exception as e:
        print("Request failed:", type(e).__name__)
        break

smart_plan = [
    p for p in smart_plan
    if not any(r["params"] == p for r in results)
]

best = min(results, key=lambda x: x["loss"])

print("Oracle calls:", len(results))
print("Remaining smart candidates:", len(smart_plan))
print("Best loss:", best["loss"])
print("Best params:", best["params"])

Oracle call: 179
Loss: 11.0055947597813
Oracle call: 180
Loss: 11.0055874721705
Oracle call: 181
Loss: 11.0055811436388
Oracle call: 182
Loss: 11.0055748219722
Oracle call: 183
Loss: 11.0055685071715
Oracle call: 184
Loss: 11.0055510964687
Oracle call: 185
Loss: 11.005544796852
Oracle call: 186
Loss: 11.0055385041032
Oracle call: 187
Loss: 11.0053593653531
Oracle call: 188
Loss: 11.0053462234356
Oracle call: 189
Loss: 11.0053220230048
Oracle call: 190
Loss: 11.0053089428535
Oracle call: 191
Loss: 11.0052958920964
Oracle call: 192
Loss: 11.0052828707386
Oracle call: 193
Loss: 11.0052698787852
Oracle call: 194
Loss: 11.0052569162416
Oracle call: 195
Loss: 11.0048758632429
Oracle call: 196
Loss: 11.0048374749164
Oracle call: 197
Loss: 11.0048102729114
Oracle call: 198
Loss: 11.004783196683
Oracle call: 199
Loss: 11.0047562462773
Oracle call: 200
Loss: 11.0047294217401
Oracle call: 201
Loss: 11.0047027231173
Oracle call: 202
Loss: 11.0046761504551
Oracle call: 203
Loss: 11.0038822314091
Or

In [62]:
for params in smart_plan[:25]:
    try:
        query_and_store(params)
    except Exception as e:
        print("Request failed:", type(e).__name__)
        break

smart_plan = [
    p for p in smart_plan
    if not any(r["params"] == p for r in results)
]

best = min(results, key=lambda x: x["loss"])

print("Oracle calls:", len(results))
print("Calls remaining to 30%:", max(0, 768 - len(results)))
print("Smart candidates remaining:", len(smart_plan))
print("Best loss:", best["loss"])
print("Best params:", best["params"])

Oracle call: 204
Loss: 11.003814091128
Oracle call: 205
Loss: 11.0037574805709
Oracle call: 206
Loss: 11.0037014080894
Oracle call: 207
Loss: 11.0036458740899
Oracle call: 208
Loss: 11.0034701240167
Oracle call: 209
Loss: 11.003415784268
Oracle call: 210
Loss: 11.0033619844664
Oracle call: 211
Loss: 11.0018444315891
Oracle call: 212
Loss: 11.0017236218708
Oracle call: 213
Loss: 11.0014855461074
Oracle call: 214
Loss: 11.0013695698096
Oracle call: 215
Loss: 11.0012558981482
Oracle call: 216
Loss: 11.0011445347253
Oracle call: 217
Loss: 11.0010354831491
Oracle call: 218
Loss: 11.0009287470333
Oracle call: 219
Loss: 10.9977739887241
Oracle call: 220
Loss: 10.9973963072802
Oracle call: 221
Loss: 10.9971458130487
Oracle call: 222
Loss: 10.9969051380765
Oracle call: 223
Loss: 10.9966743141038
Oracle call: 224
Loss: 10.9964533729762
Oracle call: 225
Loss: 10.9962423466442
Oracle call: 226
Loss: 10.9960412671645
Oracle call: 227
Loss: 10.9898716447207
Oracle call: 228
Loss: 10.9891763050429
Or

In [64]:
best_result = min(results, key=lambda x: x["loss"])

total_calls = len(results)
total_configs = 1

for values in SPACE.values():
    total_configs *= len(values)

coverage = (total_calls / total_configs) * 100

print("=" * 60)
print("FINAL OPTIMIZATION RESULT")
print("=" * 60)

print("\nBest Loss:", best_result["loss"])

print("\nBest Hyperparameters:")
for key, value in best_result["params"].items():
    print(f"{key}: {value}")

print("\nTotal Oracle Calls:", total_calls)
print("Total Search Space:", total_configs)
print("Search Space Explored: {:.2f}%".format(coverage))
print("30% Target Calls:", int(total_configs * 0.30))
print("Calls Remaining to 30%:", max(0, int(total_configs * 0.30) - total_calls))

top_results = sorted(results, key=lambda x: x["loss"])[:10]

print("\n" + "=" * 60)
print("TOP 10 CONFIGURATIONS")
print("=" * 60)

for i, r in enumerate(top_results, 1):
    print(f"\nRank {i}")
    print("Loss:", r["loss"])
    print(r["params"])

FINAL OPTIMIZATION RESULT

Best Loss: 10.9476606732259

Best Hyperparameters:
tol: 0.001
alpha: 0.06951927961775606
l1_ratio: 0.3071428571428571
positive: False
selection: cyclic
fit_intercept: True

Total Oracle Calls: 228
Total Search Space: 2560
Search Space Explored: 8.91%
30% Target Calls: 768
Calls Remaining to 30%: 540

TOP 10 CONFIGURATIONS

Rank 1
Loss: 10.9476606732259
{'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 2
Loss: 10.9477484316569
{'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 3
Loss: 10.9486038646824
{'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}

Rank 4
Loss: 10.9486672827174
{'tol': 0.0001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.05, 'positive': False, 'selection': 'cyclic', 'fit_i

In [65]:
best_result = min(results, key=lambda x: x["loss"])

print("Rank 1")
print("Loss:", best_result["loss"])
print(best_result["params"])

Rank 1
Loss: 10.9476606732259
{'tol': 0.001, 'alpha': 0.06951927961775606, 'l1_ratio': 0.3071428571428571, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}
